In [1]:
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.visualization import simple_norm

from spectral_cube import SpectralCube as sc
from regions import RectangleSkyRegion

%matplotlib widget

In [2]:
def displayimage(cube, region=None):
    img = cube.moment(order=0).value  # moment 0
    fig, ax = plt.subplots(
        figsize=(10, 2), layout="compressed", subplot_kw={"projection": cube.wcs.celestial}
    )
    norm = simple_norm(np.array(img), vmin=0, vmax=np.max(img), stretch="linear")
    im = ax.imshow(img, norm=norm, cmap="viridis", origin="lower")
    lon = ax.coords[0]
    lat = ax.coords[1]
    lon.set_major_formatter("dd")
    lat.set_major_formatter("dd")
    lon.set_axislabel("RA")
    lat.set_axislabel("Dec")
    if region != None:
        region.plot(color="white", ls="--")
    plt.colorbar(im)
    plt.show()

In [3]:
# Combine CRAFTS data cubes

file1 = "./CRAFTS_RA60_80_DEC-13_2_corrected.fits"
file2 = "./CRAFTS_RA80_100_DEC-13_2_corrected.fits"
file3 = "./CRAFTS_RA100_120_DEC-13_2_corrected.fits"
file4 = "./CRAFTS_RA120_140_DEC-13_2_corrected.fits"

In [4]:
with fits.open(file1) as hdul:
    hdr1 = hdul[0].header
    data1 = hdul[0].data
    hdul.close()

with fits.open(file2) as hdul:
    hdr2 = hdul[0].header
    data2 = hdul[0].data
    hdul.close()

with fits.open(file3) as hdul:
    hdr3 = hdul[0].header
    data3 = hdul[0].data
    hdul.close()

with fits.open(file4) as hdul:
    hdr4 = hdul[0].header
    data4 = hdul[0].data
    hdul.close()

In [5]:
# Combine the images

combined_data = np.concatenate((data4, data3, data2, data1), axis=2)
# print("Shape of combined data:", combined_data.shape)  # Should be (v, 2*y, 2*x)

In [6]:
# Update the header for the new combined image
combined_header = hdr4

combined_header["BUNIT"] = "K"
combined_header["RESTFRQ"] = 1420405751.77
combined_header["CTYPE1"] = "RA---CAR"
combined_header["CTYPE2"] = "DEC--CAR"
combined_header["CTYPE3"] = "VRAD"
combined_header['CTYPE3'] = (combined_header['CTYPE3'], 'Radio velocity (linear)')
combined_header["SPECSYS"] = "LSRK"
combined_header["VELOSYS"] = 0.0

combined_wcs = WCS(combined_header)

In [7]:
combined_cube = sc(data=combined_data, wcs=combined_wcs)
combined_cube

SpectralCube with shape=(5962, 600, 3200):
 n_x:   3200  type_x: RA---CAR  unit_x: deg    range:    60.012500 deg:  139.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:   5962  type_s: VRAD      unit_s: m / s  range:  -599897.172 m / s:  599954.336 m / s

In [9]:
combined_cube.write("CRAFTS_combined_RA60_140_DEC-13_2_corrected.fits", overwrite=True)